In [27]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

loder = PyMuPDFLoader("hackathon.pdf")

res = loder.load()

print(res)

[Document(metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'hackathon.pdf', 'file_path': 'hackathon.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'ml-based_personal_finance_optimizer', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='ML-Based Personal Finance Optimizer \n\u200b\n\u200b\n🔰 Overview\u200b\n A Personal Finance Web/Mobile App that empowers users to track their expenses, \nanalyze spending patterns using machine learning (ML), predict future financial behavior, \nand receive actionable saving recommendations tailored to their personal goals. The \nplatform will offer a seamless experience across web and mobile, with AI-generated monthly \nreports, smart alerts, and categorization of transactions. \n \n✅ Key Functional Areas \n \n👤 User Accounts & Profiles \n\u200b\nRegistration/Login via email, phone, or Google/Apple single s

In [33]:
len(res[1].page_content)

1248

In [34]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text = text_splitter.split_documents(res)
print(text)

[Document(metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'hackathon.pdf', 'file_path': 'hackathon.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'ml-based_personal_finance_optimizer', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='ML-Based Personal Finance Optimizer \n\u200b\n\u200b\n🔰 Overview\u200b\n A Personal Finance Web/Mobile App that empowers users to track their expenses, \nanalyze spending patterns using machine learning (ML), predict future financial behavior, \nand receive actionable saving recommendations tailored to their personal goals. The \nplatform will offer a seamless experience across web and mobile, with AI-generated monthly \nreports, smart alerts, and categorization of transactions. \n \n✅ Key Functional Areas'), Document(metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': 

In [35]:
len(text)

13

In [36]:
embedding = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

In [37]:
db = FAISS.from_documents(text, embedding)
db.save_local("test_db")

In [38]:
retriver = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [41]:
query="what can i add to this project to win the hackathon"
result = db.similarity_search(query=query)
print(result[0])

page_content='Security 
Encrypted storage, 2FA option, secure login 
Admin Panel 
User/account moderation, model retraining, feedback 
management 
UI/UX 
Responsive design, mobile app support, visual analytics, 
dark/light mode 
 
 
📈 Optional Enhancements 
​
Voice Entry: Input expenses via voice command. 
​
Multilingual Support: Hindi, Spanish, etc. 
​
Shared Budgets: Group budgeting with friends or family. 
​
AI Chatbot: Ask, “How much did I spend on coffee last month?” 
​' metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'hackathon.pdf', 'file_path': 'hackathon.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'ml-based_personal_finance_optimizer', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 3}


In [42]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    template="""
        Answer the question bellow and use given context for that, if you don't know the answer then simply say that "i am not sure anout that"
        
        question : {query}
        
        context : {context}
    """,
    input_variables=['query', 'context']
)

parser = StrOutputParser()

final = prompt | model | parser

print(final.invoke({'query':query, 'context': result}))

Based on the context provided, here are some optional enhancements you can consider to add to your project to potentially win the hackathon:

1. **Voice Entry**: Allow users to input expenses via voice command. This can be particularly useful for users who prefer hands-free interaction or are on the go.

2. **Multilingual Support**: Add support for languages like Hindi and Spanish. This can cater to a broader user base and demonstrate your project's global appeal.

3. **Shared Budgets**: Implement a feature that allows users to create and manage shared budgets with friends or family. This can be a valuable feature for collaborative financial planning.

4. **AI Chatbot**: Integrate an AI chatbot that can answer common financial questions, such as "How much did I spend on coffee last month?" This can provide instant feedback and enhance user engagement.

These features can differentiate your project and make it more appealing to the judges and participants.
